
# Yambda 14 · An earlier window to train the ranker on

Notebook 13 showed the user-level cross-validation was leaky: 65% of fold-B discoveries were tracks
fold-A users also discovered in the same two weeks, and without track-identifying features the
ranker's +54% fell to +5% (not significant). The ranker was memorising which tracks blew up in
the evaluation window.

Fix: train the ranker on an **earlier** window and evaluate it on the next one, exactly as it would
run in production.

| Split | Source models trained on | Ranker labels from | Role |
|---|---|---|---|
| `split_rank` (new) | days < 259 | days 259–273 | train the ranker |
| `split_val` | days < 273 | days 273–287 | evaluate the ranker, make decisions |
| `split_test` | days < 287 | days 287–301 | final check (ranker retrained on the val window) |

Same preprocessing settings as `02_preprocess`. ~4 min. Produces `colab_rank.zip` for notebook 15.


In [1]:

import zipfile
from musicrec import config
from musicrec.yambda import build_events
from musicrec.preprocess import make_split, extra_targets

events = build_events(min_played_ratio=50)
s = make_split(events, config.YAMBDA_RANK_TRAIN_END, config.YAMBDA_RANK_EVAL_END,
               min_item_users=5, min_user_items=10, positive="is_full_listen", target="is_organic")
s.save(config.YAMBDA_SPLIT_RANK_DIR)
extra_targets(events, s, positive="is_full_listen").to_parquet(config.YAMBDA_SPLIT_RANK_DIR / "targets_all.parquet", index=False)
s.stats


{'n_users': 8310,
 'n_items': 153565,
 'train_interactions': 22262012,
 'train_pairs': 6799529,
 'density': 0.005328261887166788,
 'eval_new_pairs_all': 145338,
 'eval_new_pairs_reachable': 99909,
 'eval_users_with_targets': 5450}

In [2]:

overlap = s.targets.merge(s.train_counts, on=["user_idx", "item_idx"])
assert overlap.empty and s.train_events["ts"].max() < s.train_end
print("checks passed")

# bundle for Colab (SASRec needs this split)
root = config.PROJECT_ROOT
with zipfile.ZipFile(root / "colab_rank.zip", "w", zipfile.ZIP_STORED) as z:
    for f in config.YAMBDA_SPLIT_RANK_DIR.iterdir():
        z.write(f, f.relative_to(root))
print(round((root / "colab_rank.zip").stat().st_size / 1e6), "MB")


checks passed
180 MB
